# 4. Train CatBoost (optional)

Trains a CatBoost classifier on **`ground_truth`** for each schema and
predicts every invoice, writing `source='catboost'` rows into
`predictions` with a real `predict_proba` confidence.

This is the supervised half of the feedback loop: it only works once there
is enough ground truth (bulk-imported or grown from review), and it gets
better as ground truth grows. Skipping this notebook never breaks the
pipeline — the LLM classifier (notebook 3) already labels every invoice.

Tested on Serverless v4.

In [ ]:
%pip install pyyaml openpyxl pydantic catboost scikit-learn
%restart_python

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_spark
from src.config import load_config
from src.schemas import list_schemas
from src.categorize import train_catboost

spark = get_spark()
config = load_config()

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', config.catalog)
dbutils.widgets.text('schema', config.schema_name)
dbutils.widgets.text('invoices_table', config.invoices)
dbutils.widgets.text('run_catboost', 'true')
dbutils.widgets.text('min_rows', '50')   # minimum ground-truth rows to train

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
invoices_table = dbutils.widgets.get('invoices_table')
run_catboost = dbutils.widgets.get('run_catboost').lower() == 'true'
min_rows = int(dbutils.widgets.get('min_rows'))
spark.sql(f'USE {catalog}.{schema}')

## Train on ground truth and predict every invoice

`train_catboost` trains one model on `ground_truth` for the schema
(predicting the leaf `code` directly — schema-agnostic, no per-level
models), predicts every invoice, looks up each predicted code's label and
`level_path` from the taxonomy, and MERGEs `source='catboost'` rows into
`predictions`. It returns 0 and skips cleanly when there isn't enough
ground truth.

In [ ]:
if run_catboost:
    for s in list_schemas():
        print(f'\n==> {s.name}')
        train_catboost(
            spark, s,
            catalog=catalog, schema_db=schema,
            invoices_table=invoices_table,
            min_rows=min_rows,
        )
else:
    print('Skipping CatBoost (run_catboost=false)')